In [1]:
# ════════════════════════════════════════════════════════════════════════════════════════════
# ScoutIA — MONTAGE ET FICHE D'UN JOUEUR DE BAELEN (sans refaire l'analyse du match) — 24/11/2026
# Joueur : n° 6, équipe BLEU CLAIR (P4). Utilise le module de production analysis/montage_joueur.py, comme le pipeline.
# ENTRÉES (Add Input) :
#   • la vidéo de Baelen (dataset baelen : p4-baelen-2026-09-12.mp4) ;
#   • la SORTIE du notebook de l'analyse de Baelen (Notebook Output) : cache_baelen_reference.zip,
#     bout_en_bout_baelen.json, gemini_cache_baelen.jsonl, rapport_baelen*.pdf ;
#   • codeprojectai (parseq_scoutia_maillots_v2.pt) ; mziparameter (param.env, nouvelle clé Gemini).
# GitHub à jour : analysis/montage_joueur.py, analysis/montage_joueur_script.py, export/fiche_joueur.py,
#   export/rapport_client.py, export/resume_match.py.
# ACCÉLÉRATEUR : GPU T4 conseillé — les numéros de Baelen n'ont jamais été lus (pas de fichier de lectures) : la lecture
#   PARSeq se fait ici (≈ 1 h 30 à 2 h 30 sans GPU, estimation). Coût Gemini estimé ≈ 0,25 à 0,35 $. Save & Run All.
# À GARDER : joueur_baelen_6_bleu_clair.zip (montage, fiche, séquences pour « Composer son montage », lectures des
#   numéros réutilisables pour un autre joueur de Baelen).
# ════════════════════════════════════════════════════════════════════════════════════════════
MATCH = "baelen"
JOUEURS = [dict(numero=6, couleur="bleu clair")]

import glob, json, os, shutil, subprocess, sys, zipfile

REPO = "/kaggle/working/ProjectAIMika"
if not os.path.isdir(REPO):
    if os.system(f"git clone -q --depth 1 https://github.com/mikarocco-droid/ProjectAIMika.git {REPO}") != 0:
        raise SystemExit("❌ clone du dépôt impossible")
for f_ in ("analysis/montage_joueur.py", "analysis/montage_joueur_script.py", "export/fiche_joueur.py"):
    if not os.path.exists(f"{REPO}/{f_}"):
        raise SystemExit(f"❌ {f_} absent de GitHub")
if "COMPOSER SON MONTAGE" not in open(f"{REPO}/analysis/montage_joueur_script.py", encoding="utf-8").read():
    print("⚠️ analysis/montage_joueur_script.py n'est pas la dernière version (pas de « Composer son montage ») : "
          "mettez-la sur GitHub avant de lancer.")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "timm", "pytorch-lightning", "nltk", "fpdf2", "pypdf",
                "google-genai", "opencv-python-headless"], check=False)
sys.path.insert(0, REPO)


def _un(motif, quoi, obligatoire=True):
    r = sorted(glob.glob(f"/kaggle/input/**/{motif}", recursive=True))
    if not r and obligatoire:
        raise SystemExit(f"❌ {quoi} introuvable dans les entrées")
    return r[0] if r else None


video = _un("p4-baelen-2026-09-12.mp4", "la vidéo de Baelen")
relais = json.load(open(_un(f"bout_en_bout_{MATCH}.json", f"bout_en_bout_{MATCH}.json"), encoding="utf-8"))
_un(f"gemini_cache_{MATCH}*.jsonl", "les réponses Gemini de l'analyse de Baelen")
poids = _un("parseq_scoutia_maillots_v2.pt", "parseq_scoutia_maillots_v2.pt (codeprojectai)")
rapport = next((r for r in sorted(glob.glob(f"/kaggle/input/**/rapport_{MATCH}*.pdf", recursive=True)) if "_joueur" not in r), None)

# cache du suivi : archive zip → dossier
cache = "/kaggle/working/cache_baelen"
if not os.path.exists(f"{cache}/frames_data.pkl"):
    cz = _un(f"cache_{MATCH}_reference*.zip", f"cache_{MATCH}_reference.zip")
    os.makedirs(cache, exist_ok=True)
    with zipfile.ZipFile(cz) as z:
        for n in z.namelist():
            if n.endswith("/"):
                continue
            with z.open(n) as src, open(os.path.join(cache, os.path.basename(n)), "wb") as dst:
                shutil.copyfileobj(src, dst)
print(f"cache du suivi : {cache} ({', '.join(sorted(os.listdir(cache)))})")

cle = ""
pe = _un("mziparameter/**/param.env", "param.env (mziparameter)", obligatoire=False) or _un("param.env", "param.env", False)
for l in open(pe, encoding="utf-8") if pe else []:
    if l.strip().startswith("GEMINI_API_KEY="):
        cle = l.split("=", 1)[1].strip().strip('"').strip("'")
if not cle:
    raise SystemExit("❌ clé Gemini absente de param.env")

import torch
print(f"GPU : {'oui (' + torch.cuda.get_device_name(0) + ')' if torch.cuda.is_available() else 'NON — lecture des numéros plus lente'}")

from analysis.montage_joueur import monter_joueur

for j in JOUEURS:
    nom = f"joueur_{MATCH}_{j['numero']}_{j['couleur'].replace(' ', '_')}"
    sortie = f"/kaggle/working/{nom}"
    print(f"\n{'═' * 90}\nJOUEUR n°{j['numero']} {j['couleur']}{' (gardien)' if j.get('gardien') else ''} — {MATCH}\n{'═' * 90}", flush=True)
    res = monter_joueur(video=video, cache_dir=cache, sortie=sortie, match=MATCH, bornes=relais.get("bornes"),
                        buts=relais.get("buts"), entrees=["/kaggle/input", "/kaggle/working"], gemini_key=cle,
                        parseq_poids=poids, rapport_pdf=rapport, **j)
    for k in ("trouve", "erreur", "message", "presence_s", "passages", "moments_ballon", "buts_confirmes",
              "buts_possibles", "buts_impliques", "arrets", "tirs_adverses", "cout_gemini_usd", "duree_min", "montage",
              "fiche", "fiche_donnees", "sequences_joueur", "rapport_et_fiche", "journal"):
        if k in res:
            print(f"   {k:18s} : {res[k]}")
    shutil.make_archive(f"/kaggle/working/{nom}", "zip", "/kaggle/working", nom)
    print(f"\n✅ À télécharger : /kaggle/working/{nom}.zip")

print("\nPour la démonstration : dézippez l'archive, puis "
      "python outils/demo_resultat.py --dossier demo_baelen --email … --titre \"P4 vs Baelen\" --joueur joueur_baelen_6_bleu_clair")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.2/91.2 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 360.6/360.6 kB 10.5 MB/s eta 0:00:00
cache du suivi : /kaggle/working/cache_baelen (events.pkl, frames_data.pkl, jersey_map.pkl, meta.json, team_colors.pkl)
GPU : oui (Tesla T4)

══════════════════════════════════════════════════════════════════════════════════════════
JOUEUR n°6 bleu clair — baelen
══════════════════════════════════════════════════════════════════════════════════════════
   relais : fourni par le pipeline
   BOUT EN BOUT : baelen | cache neuf /kaggle/working/cache_baelen | 0 action(s) du résumé produites par le pipeline
✅ Cache : /kaggle/working/cache_baelen
   coup d'envoi 05:35 | 48575 événements | 35334 images suivies | 0 numéros lus
✅ time_since_update présent (100.0 % des positions) : seules les vraies détections (== 0) sont utilisées
   5667 pistes DeepSort → 37841 segments | coupures : {'trou': 32158, 'taille': 16} | positions prédites éca